# 08 — Refresh Hiring Analytics Dashboard

**J&J HRD 2030 Predictive Hiring Demo**

Final pipeline step: finds the deployed Lakeview dashboard and triggers a dataset refresh
so stakeholders see current data when the pipeline completes.

### What this notebook does
1. Looks up the `J&J HRD 2030 — Hiring Analytics` dashboard via the REST API
2. Triggers a published dashboard refresh
3. Prints the dashboard URL for stakeholders

### Prerequisites
- `databricks bundle deploy` has been run (dashboard is deployed to workspace)
- Notebooks `00` – `07` have completed successfully

**This is the final notebook in the pipeline.**

## Setup

Configure widgets, resolve workspace host and token, and set the target dashboard name.

In [ ]:
dbutils.widgets.text("catalog",        "bx4",      "UC Catalog")
dbutils.widgets.text("schema",         "hrd_2030", "UC Schema")
dbutils.widgets.text("warehouse_id",   "",         "SQL Warehouse ID")
dbutils.widgets.text("dashboard_name", "J&J HRD 2030 — Hiring Analytics", "Dashboard Display Name")

In [ ]:
import requests, time

catalog        = dbutils.widgets.get("catalog")
schema         = dbutils.widgets.get("schema")
warehouse_id   = dbutils.widgets.get("warehouse_id")
dashboard_name = dbutils.widgets.get("dashboard_name")

ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()
host  = ctx.apiUrl().get().rstrip("/")
hdrs  = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"Catalog        : {catalog}")
print(f"Schema         : {schema}")
print(f"Warehouse ID   : {warehouse_id or '(not set)'}")
print(f"Dashboard Name : {dashboard_name}")
print(f"Host           : {host}")

## Find Dashboard by Name

List all Lakeview dashboards via REST API and locate the J&J HRD 2030 dashboard by its display name.

In [ ]:
# ---------------------------------------------------------------------------
# Find the dashboard by display name via the Lakeview REST API
# ---------------------------------------------------------------------------
dashboard_id = None
dashboard_url = None

resp = requests.get(
    f"{host}/api/2.0/lakeview/dashboards",
    headers=hdrs,
    params={"page_size": 100},
    timeout=30,
)

if resp.status_code == 200:
    dashboards = resp.json().get("dashboards", [])
    for d in dashboards:
        if d.get("display_name", "") == dashboard_name:
            dashboard_id  = d["dashboard_id"]
            dashboard_url = f"{host}/dashboardsv3/{dashboard_id}"
            print(f"✅ Found dashboard: {dashboard_name}")
            print(f"   Dashboard ID  : {dashboard_id}")
            print(f"   Dashboard URL : {dashboard_url}")
            break

    if not dashboard_id:
        print(f"⚠️  Dashboard '{dashboard_name}' not found in workspace.")
        print(f"   Available dashboards ({len(dashboards)}):")
        for d in dashboards[:10]:
            print(f"     - {d.get('display_name')} ({d.get('dashboard_id')})")
else:
    print(f"⚠️  Could not list dashboards: HTTP {resp.status_code} — {resp.text[:300]}")

## Inject Catalog / Schema

If the dashboard definition contains `__CATALOG__`/`__SCHEMA__` placeholders, replace them with the runtime catalog and schema values.

In [ ]:
# ---------------------------------------------------------------------------
# Inject catalog/schema into dashboard definition (replace __CATALOG__/__SCHEMA__)
# This allows the dashboard JSON to ship with placeholders and be resolved at
# pipeline runtime, so the bundle file stays environment-agnostic.
# ---------------------------------------------------------------------------
if dashboard_id:
    get_resp = requests.get(
        f"{host}/api/2.0/lakeview/dashboards/{dashboard_id}",
        headers=hdrs,
        timeout=30,
    )
    if get_resp.status_code == 200:
        dash_def      = get_resp.json()
        serialized    = dash_def.get("serialized_dashboard", "")
        warehouse_val = dash_def.get("warehouse_id", warehouse_id)

        if "__CATALOG__" in serialized or "__SCHEMA__" in serialized:
            patched = serialized.replace("__CATALOG__", catalog).replace("__SCHEMA__", schema)

            put_resp = requests.put(
                f"{host}/api/2.0/lakeview/dashboards/{dashboard_id}",
                headers=hdrs,
                json={
                    "display_name":        dashboard_name,
                    "serialized_dashboard": patched,
                    "warehouse_id":        warehouse_val or warehouse_id,
                },
                timeout=30,
            )
            if put_resp.status_code in (200, 201):
                print(f"✅ Dashboard updated: catalog={catalog}, schema={schema}")
            else:
                print(f"⚠️  Failed to patch dashboard: HTTP {put_resp.status_code} — {put_resp.text[:300]}")
        else:
            print(f"ℹ️  No placeholders found in dashboard definition — skipping patch.")
    else:
        print(f"⚠️  Could not fetch dashboard definition: HTTP {get_resp.status_code} — {get_resp.text[:200]}")
else:
    print("Skipping catalog/schema injection — dashboard not found.")

## Trigger Dashboard Refresh

Publish a dataset refresh so the dashboard shows current data from the Gold table when the pipeline completes.

In [ ]:
# ---------------------------------------------------------------------------
# Trigger a dataset refresh on the published dashboard
# ---------------------------------------------------------------------------
if not dashboard_id:
    print("Skipping refresh — dashboard not found.")
else:
    refresh_resp = requests.post(
        f"{host}/api/2.0/lakeview/dashboards/{dashboard_id}/published",
        headers=hdrs,
        json={"warehouse_id": warehouse_id} if warehouse_id else {},
        timeout=30,
    )
    print(f"Refresh request: HTTP {refresh_resp.status_code}")
    if refresh_resp.status_code in (200, 201, 204):
        print(f"✅ Dashboard dataset refresh triggered.")
    else:
        print(f"   Response: {refresh_resp.text[:300]}")
        print("   (Non-critical — dashboard data is still current from the Gold table)")

## Pipeline Complete

Print a summary of all deployed resources — Gold table, Genie Space, ML endpoint, Agent endpoint, and Dashboard URL.

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline complete — print stakeholder summary
# ---------------------------------------------------------------------------
print("=" * 60)
print("J&J HRD 2030 — Pipeline Complete")
print("=" * 60)
print()
print(f"Gold table   : {catalog}.{schema}.candidate_scoring_summary")
print(f"Genie Space  : Create in UI if not auto-created")
print(f"ML Endpoint  : hire-right-model-endpoint")
print(f"Agent        : hire-right-agent-endpoint")
if dashboard_url:
    print(f"Dashboard    : {dashboard_url}")
else:
    print(f"Dashboard    : Deploy with 'databricks bundle deploy' then check workspace")
print()
print("✅ All pipeline tasks complete.")